# VisionOps Guard - Kaggle Notebook Training Pipeline 🏆
**Objective:** Train Safety PPE Model on Kaggle GPU / Dual-GPU Environment.

> **Important Kaggle Checklist:**
> 1. **Accelerator:** Ensure `GPU T4 x2` or `GPU P100` is selected in the right panel (`Notebook options` -> `Accelerator`).
> 2. **Internet:** Ensure the toggle **"Internet ON"** is enabled to permit package installation.

In [ ]:
# 1. Verify Kaggle GPU Environment
!nvidia-smi
import torch
gpu_count = torch.cuda.device_count()
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    print(f"    GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 2. Install Ultralytics & ONNX Tools
!pip install --quiet ultralytics onnx onnxruntime albumentations pyyaml

In [ ]:
# 3. Configure Kaggle Working Directories & Dataset
import os
import yaml

# Kaggle writes MUST be placed in /kaggle/working
WORKING_DIR = '/kaggle/working/visionops-guard'
os.makedirs(f"{WORKING_DIR}/models", exist_ok=True)

# Configure Dataset for Kaggle
kaggle_dataset_yaml = {
    'path': f"{WORKING_DIR}/data/processed",
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'person',
        1: 'hardhat',
        2: 'vest',
        3: 'no_hardhat',
        4: 'no_vest'
    }
}

dataset_yaml_path = f"{WORKING_DIR}/dataset_kaggle.yaml"
with open(dataset_yaml_path, 'w') as f:
    yaml.dump(kaggle_dataset_yaml, f)
print(f"[+] Kaggle dataset configuration saved at: {dataset_yaml_path}")

In [ ]:
# 4. Launch Training with Dual-GPU Support
from ultralytics import YOLO

# Select single or dual GPU based on Kaggle hardware setting
active_devices = [0, 1] if gpu_count >= 2 else 0
print(f"[*] Training on Device(s): {active_devices}")

model = YOLO('yolov8n.pt')

results = model.train(
    data=dataset_yaml_path,
    epochs=25,
    imgsz=640,
    batch=16 * (gpu_count if gpu_count > 0 else 1),
    device=active_devices,
    project='/kaggle/working/runs',
    name='visionops_kaggle'
)

In [ ]:
# 5. Export to ONNX & Save to Kaggle Output Root
import shutil

best_pt = '/kaggle/working/runs/visionops_kaggle/weights/best.pt'
trained_model = YOLO(best_pt)
onnx_export = trained_model.export(format='onnx', imgsz=640, simplify=True)

# Copy to /kaggle/working/ root so they appear immediately in Kaggle Output Viewer
shutil.copy(best_pt, '/kaggle/working/best_model.pt')
shutil.copy(onnx_export, '/kaggle/working/best_model.onnx')

print("[+] Model export complete! You can download 'best_model.pt' & 'best_model.onnx' directly from the Kaggle Output Tab.")